# Lab: Pydantic as an LLM firewall
Feed 20 malformed 'LLM outputs' through a schema. Expect >= 95% rejected.

In [ ]:
!pip install -q pydantic

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError, field_validator

class Classification(BaseModel):
    category: Literal['billing','bug','feature','spam']
    confidence: float = Field(ge=0, le=1)
    reasoning: str = Field(min_length=10, max_length=500)

    @field_validator('category', mode='before')
    @classmethod
    def normalize(cls, v):
        return v.strip().lower() if isinstance(v, str) else v

def try_parse(raw):
    try:
        return Classification.model_validate_json(raw)
    except ValidationError:
        return None

samples = [
    '{"category":"Bug ","confidence":0.9,"reasoning":"real stack trace here xxxxx"}',
    '{"category":"urgent","confidence":0.9,"reasoning":"bad literal xxxxxxxx"}',
    '{"category":"bug","confidence":1.4,"reasoning":"range xxxxxxxxxxxx"}',
    '{"category":"bug","confidence":0.9}',
    'not json',
]
parsed = [try_parse(s) for s in samples]
print('accepted:', sum(p is not None for p in parsed), '/', len(samples))

No LLM call. cost = $0.00 / run (cached sample strings).